# (0) Setup database

You need to point towards the "credentials.txt" file which has the following structure:

```
username=your_email@example.com
password=your_ecoinvent_password
```

In [1]:
from pathlib import Path
import bw2data as bd
import bw2io as bi

PROJECT = "ammonia_final"
DB_NAME = "ecoinvent-3.10-cutoff"
CRED_PATH = Path("credentials.txt")

def read_credentials(path: Path):
    if not path.is_file():
        raise FileNotFoundError(f"Couldn't find credentials file at: {path.resolve()}")
    creds = {}
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        # allow "key=value" or "key: value" or "key value"
        for sep in ("=", ":", " "):
            if sep in line:
                k, v = line.split(sep, 1)
                creds[k.strip().lower()] = v.strip()
                break
    if "username" not in creds or "password" not in creds:
        raise ValueError("credentials.txt must contain 'username' and 'password'.")
    return creds["username"], creds["password"]

# 1) Ensure project exists / is selected
bd.projects.set_current(PROJECT)

# 2) Import ecoinvent 3.10 cutoff if missing
if DB_NAME in bd.databases:
    print(f"Database '{DB_NAME}' already exists in project '{bd.projects.current}'.")
else:
    username, password = read_credentials(CRED_PATH)
    bi.import_ecoinvent_release(
        version="3.10",
        system_model="cutoff",  # "cutoff", "apos", "consequential", or "EN15804"
        username=username,
        password=password,
    )
    print(f"Database '{DB_NAME}' installed successfully.")

Database 'ecoinvent-3.10-cutoff' already exists in project 'ammonia_final'.


In [2]:
# Path to your Excel file
excel_path = r"ammonia.xlsx"
fg_db_name = "ammonia"

if fg_db_name in bd.databases:
    print(f"Database '{fg_db_name}' already exists in project '{bd.projects.current}'.")
else:
    # 1. Import the Excel file
    fg_db = bi.ExcelImporter(excel_path)

    # 2. Apply strategies to clean and prepare the data
    fg_db.apply_strategies()

    # 3. Match the foreground database to itself (for internal links)
    fg_db.match_database(fields=["name", "unit", "reference product", "location"])

    # 4. Match to ecoinvent technosphere (use your actual ecoinvent db name)
    fg_db.match_database(
        "ecoinvent-3.10-cutoff",
        fields=["name", "unit", "location", "reference product"]
    )

    # 5. Match to ecoinvent biosphere (biosphere db is usually auto-named, check with list(bd.databases))
    biosphere_db = [db for db in bd.databases if "biosphere" in db and "3.10" in db][0]
    fg_db.match_database(
        biosphere_db,
        fields=["name", "categories", "location"]
    )

    # 6. Check statistics (should have 0 unlinked exchanges)
    fg_db.statistics()

    # 7. Write the database to disk
    fg_db.write_database()
    print(f"Database '{fg_db_name}' installed successfully.")

Extracted 1 worksheets in 0.03 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 2.73 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields


Writing activities to SQLite3 database:


46 datasets
519 exchanges
0 unlinked exchanges
  


0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Title: Writing activities to SQLite3 database:
  Started: 09/08/2025 16:58:11
  Finished: 09/08/2025 16:58:11
  Total time elapsed: 00:00:00
  CPU %: 133.00
  Memory %: 0.91
Created database: ammonia
Database 'ammonia' installed successfully.


# (1) Solve Base Case Study

In [3]:
# Streamlined Ammonia Case Study - Minimal Output
import pandas as pd
import numpy as np
from pulpo import pulpo

# --- Config ---
project = "ammonia_reduced_unc"
database = "ammonia"
method = "('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')"

# --- Initialize ---
pulpo_worker = pulpo.PulpoOptimizer(project, database, method, directory="develop_tests")
pulpo_worker.intervention_matrix = "ecoinvent-3.10-biosphere"
pulpo_worker.get_lci_data()

# --- Deterministic process retrieval (avoid order issues) ---
def get_single_process(worker, query, prefer_locations=("RER", "Europe", "GLO")):
    matches = worker.retrieve_processes(processes=query)
    if not matches:
        raise ValueError(f"No process found for query: {query}")
    for loc in prefer_locations:
        for p in matches:
            if getattr(p, "location", None) == loc or loc in str(p):
                return p
    return sorted(matches, key=lambda x: str(x))[0]

# --- Choice definitions (capacities bound per-label) ---
choice_config = {
    "biogas": {
        "processes": [
            "anaerobic digestion of agricultural residues",
            "anaerobic digestion of sequential crop",
        ],
        # 2030 EU-27 potentials from biomethane shares (38 bcm total; 24% ag, 21% sequential),
        # converted to raw biogas assuming ~57% CH₄ → 16.0 & 14.0 bcm ≈ 1.60e10 & 1.40e10 m³/yr.
        "capacities": [1.60e10, 1.40e10],
    },
    "biomethane": {
        "processes": [
            "upgrading water scrubbing (CCS)",
            "upgrading water scrubbing",
            "upgrading chemical scrubbing",
            "upgrading chemical scrubbing (CCS)",
        ],
        "capacities": [1e20, 1e20, 1e20, 1e20],
    },
    "methane": {
        "processes": ["market for methane fg", "market for biomethane"],
        "capacities": [1e20, 1e20],
    },
    "heat": {
        "processes": ["heat from methane", "heat from methane (CCS)", "heat from hydrogen"],
        "capacities": [1e20, 1e20, 1e20],
    },
    "hydrogen": {
        "processes": [
            "methane pyrolysis",
            "steam methane reforming",
            "steam methane reforming (CCS)",
            "plastics gasification",
            "plastics gasification (CCS)",
            "alkaline electrolysis",
            "PEM electrolysis",
        ],
        # Methane pyrolysis capped to 10,000 t H2/yr (= 1.0e7 kg/yr); others left high for now.
        "capacities": [1.0e7, 1e20, 1e20, 1e20, 1e20, 1e20, 1e20],
    },
    "electricity": {
        "processes": [
            "grid electricity",
            "wind onshore electricity",
        ],
        "capacities": [1e20, 1e10], # Placeholder cap for wind onshore
    },
    "ammonia": {
        "processes": [
            "steam reforming, integrated",
            "steam reforming, integrated (CCS)",
            "nitrogen + hydrogen",
        ],
        "capacities": [1e20, 1e20, 1e20],
    },
}

# --- Build choices with deterministic mapping ---
choices = {}
for category, cfg in choice_config.items():
    labels, caps = cfg["processes"], cfg["capacities"]
    if len(labels) != len(caps):
        raise ValueError(f"Length mismatch in '{category}': {len(labels)} labels vs {len(caps)} capacities")
    choices[category] = {get_single_process(pulpo_worker, lbl): cap for lbl, cap in zip(labels, caps)}

# --- Demand (EU ammonia, kg/yr) ---
demand_process = get_single_process(pulpo_worker, "market for ammonia")
demand = {demand_process: 17.1e9}  # ~17.1 Mt/yr (EU), set to taste for your scenario

# --- Additional upper bounds (shared resources / feedstocks) ---
waste_pp = get_single_process(pulpo_worker, "treatment of waste PP")
waste_ps = get_single_process(pulpo_worker, "treatment of waste PS")
ccs_process = get_single_process(pulpo_worker, "CCS 200km pipeline 1000m deep")

upper_bounds = {
    waste_pp: 1.875e9,  # 25% of ~7.5 Mt PP post-consumer waste ≈ 1.875 Mt/yr
    waste_ps: 3.25e8,   # 25% of ~1.3 Mt PS waste ≈ 0.325 Mt/yr
    ccs_process: 5.0e9, # 5 MtCO2/yr (10% of EU-27 2030 NZIA target)
}

# --- Solve ---
pulpo_worker.instantiate(demand=demand, choices=choices, upper_limit=upper_bounds)
pulpo_worker.solve()
pulpo_worker.summarize_results(zeroes=True)

print(f"✅ Setup complete: {sum(len(c) for c in choices.values())} alternatives across {len(choices)} categories")


Creating Instance
Instance created
optimal solution found:  18447328441.3132


## Total Impact(s)

,Weight,Value
Method,,
"('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')",1,1.844733e+10


## Choices Made

### biogas

...

,Value,Capacity
Metadata,,
nitrogen + hydrogen | ammonia | RER,1.710000e+10,1.000000e+20


## Constraints

### Constraints Upper

,Key,Metadata,Value,Limit
ID,,,,
23579,"(ammonia, fc378c5b9e61e417e77ba0c166897da5_copy1)",CCS 200km pipeline 1000m deep | CO2 stored | RER,5.000000e+09,5.000000e+09


✅ Setup complete: 23 alternatives across 7 categories
